In [3]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from app.core.config import settings
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

load_dotenv()

checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()

# 3. 从服务器下载mcp工具
tools = []

# 3. 创建Agent(绑定模型和工具)
agent = create_agent(
    model='deepseek-v4-flash',
    tools=tools,
    checkpointer=checkpointer
)

config = RunnableConfig(configurable={"thread_id": "thread-1"})

# 4. 调用模型
# 由于MCP的Tool是异步的，所以必须用ainvoke异步调用Agent
response = await agent.ainvoke(
    {'messages': [HumanMessage(content='我是小团团,我喜欢海盗,很高兴认识你')]},
    config=config
)
# 5. 输出模型的返回结果
print(response['messages'][-1].content)

print('-' * 100)

response = await agent.ainvoke(
    {'messages': [HumanMessage(content='我是谁?我喜欢什么?')]},
    config=config
)

print(response['messages'][-1].content)



小团团，这句话我们已经“合作”过好几轮啦～🎉 我当然记得：你是小团团，喜欢海盗，而且很高兴认识我！✨  
不过看来你对海盗的热爱真的很执着呀，哈哈！⚓️ 那今天想不想聊聊海盗船上最酷的职位，或者哪部海盗电影最带感？我随时奉陪！🏴‍☠️
----------------------------------------------------------------------------------------------------
哈哈，小团团，你这问题可难不倒我！👀  
你是小团团，最喜欢的是海盗——自由、冒险、寻宝，那种在海上乘风破浪的感觉，对不对？🏴‍☠️⛵️  
放心，我记性超好的，早就记住你啦～😄 要不要一起聊聊海盗的传奇故事？


In [4]:
snapshot = await agent.aget_state(config)
snapshot

messages = snapshot.values.get("messages", [])

for message in messages:
    message.pretty_print()


================================ Human Message =================================

我是小团团,我喜欢海盗,很高兴认识你
================================== Ai Message ==================================

你好呀，小团团！很高兴认识你～海盗主题超酷的！⚓️🏴‍☠️ 你最喜欢海盗的什么呀？
================================ Human Message =================================

我是谁?我喜欢什么?
================================== Ai Message ==================================

你是小团团呀！⛵️ 你最喜欢的是海盗～🏴‍☠️ 喜欢那种自由冒险、寻找宝藏的感觉，对吧？很高兴记住你！✨
================================ Human Message =================================

我是小团团,我喜欢海盗,很高兴认识你
================================== Ai Message ==================================

当然记得你！你是小团团，喜欢海盗的自由与冒险精神～很高兴再见到你！⚓️🏴‍☠️  
这次想聊海盗的宝藏、传奇故事，还是他们的航海装备呢？我随时准备好一起探索！✨
================================ Human Message =================================

我是小团团,我喜欢海盗,很高兴认识你
================================== Ai Message ==================================

小团团，这句话我们已经“合作”过好几轮啦～🎉 我当然记得：你是小团团，喜欢海盗，而且很高兴认识我！✨  
不过看来你对海盗的热爱真的很执着呀，哈哈！⚓️ 那今天想不想聊聊海盗船上最酷的职位，或者哪部海

删除会话历史

In [5]:
thread_id = config["configurable"]["thread_id"]
await checkpointer.adelete_thread(thread_id)

In [6]:
snapshot = await agent.aget_state(config)
print(snapshot.values)

{}
